<a href="https://colab.research.google.com/github/brpetros/prompts_and_evalution_notebooks/blob/main/3_autorubric_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import json
from pprint import pprint
import os
from google.colab import userdata


os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
# get responses from file and save a json form
def get_responses(file_path):
    responses = []

    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line.strip())
            if data is not None: responses.append(data)

    # saves the output in json form for easy iteration [we still keep the text form for llm evaluation]
    for response in responses:
      if response != None:
        response["output"]["json"] = json.loads(response["output"]["text"])

    return responses

## sample responses

In [ ]:
def sample_responses(responses):
    sample_df = pd.read_csv('/content/notion_evaluation_sample.csv')
    evaluation_sample = sample_df[['Name']].values.flatten().tolist()
    return [response for response in responses if response["output"]["json"]["skill_name"].lower() in evaluation_sample]

## Evaluation using AutoRubric and LLM

In [ ]:
!pip install autorubric

In [ ]:
from autorubric import RubricDataset, LLMConfig, evaluate, DataItem, Rubric, EvalRunner, EvalConfig, EvalResult, CannotAssessConfig, CannotAssessStrategy
from autorubric.graders import CriterionGrader
import uuid
import time
from datetime import datetime
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
rubric_criteria = [
    {
        "name": "accuracy",
        "requirement": "The response is factually correct, free from hallucinations, and aligns with domain knowledge.",
        "options": [
            {
                "label": "Contains inaccurate information or hallucinations",
                "value": 0.0
            },
            {
                "label": "Fully accurate and factually correct",
                "value": 1.0
            }
        ]
    },
    {
        "name": "fluency",
        "requirement": "The output is well-formed, grammatically and syntactically correct.",
        "scale_type": "ordinal",
        "options": [
            {
                "label": "Unreadable or frequent grammatical errors severely affecting comprehension",
                "value": 0.0
            },
            {
                "label": "Frequent errors; meaning is often unclear or awkward",
                "value": 0.25
            },
            {
                "label": "Understandable but with noticeable grammatical or syntactic issues",
                "value": 0.5
            },
            {
                "label": "Mostly fluent with minor language issues",
                "value": 0.75
            },
            {
                "label": "Fully fluent, grammatically correct, and natural-sounding text",
                "value": 1.0
            }
        ]
    },
    {
        "name": "relevance",
        "requirement": "The response addresses the user’s query, avoiding off-topic or unnecessary information.",
        "scale_type": "ordinal",
        "options": [
            {
                "label": "Completely off-topic or unrelated to the query",
                "value": 0.0
            },
            {
                "label": "Mostly irrelevant with minimal useful content",
                "value": 0.25
            },
            {
                "label": "Partially relevant but includes significant irrelevant information",
                "value": 0.5
            },
            {
                "label": "Mostly relevant with minor digressions",
                "value": 0.75
            },
            {
                "label": "Fully relevant and focused on the query",
                "value": 1.0
            }
        ]
    },
    {
        "name": "clarity",
        "requirement": "The response is easy to understand, logically structured, and presents ideas in a coherent and well-organized manner.",
        "scale_type": "ordinal",
        "options": [
            {
                "label": "Confusing, poorly structured, or hard to follow",
                "value": 0.0
            },
            {
                "label": "Mostly unclear with weak structure and disjointed ideas",
                "value": 0.25
            },
            {
                "label": "Understandable but requires effort; some structural or logical issues",
                "value": 0.5
            },
            {
                "label": "Clear and well-structured but with minor issues",
                "value": 0.75
            },
            {
                "label": "Very clear, logically structured, and easy to follow throughout",
                "value": 1.0
            }
        ]
    },
    {
        "name": "depth",
        "requirement": "The response provides meaningful insight beyond surface-level information, including explanation, reasoning, or elaboration when appropriate.",
        "scale_type": "ordinal",
        "options": [
            {
                "label": "Surface-level and too general response",
                "value": 0.0
            },
            {
                "label": "Mostly superficial",
                "value": 0.25
            },
            {
                "label": "Some content is general and surface-level",
                "value": 0.5
            },
            {
                "label": "Mostly deep and explanatory but with minor presence of superficial or general content",
                "value": 0.75
            },
            {
                "label": "Fully insightful and explanatory",
                "value": 1.0
            }
        ]
    },
    {
        "name": "consistency",
        "requirement": "The response remains internally non-contradictory and maintains a stable tone, role, and stance throughout.",
        "scale_type": "ordinal",
        "options": [
            {
                "label": "Completely inconsistent tone/role with contradictions",
                "value": 0.0
            },
            {
                "label": "Mostly inconsistent with frequent issues",
                "value": 0.25
            },
            {
                "label": "Partially consistent with noticeable inconsistencies",
                "value": 0.5
            },
            {
                "label": "Mostly consistent with minor issues",
                "value": 0.75
            },
            {
                "label": "Fully consistent, no contradictions, stable tone and role",
                "value": 1.0
            }
        ]
    }
]

In [ ]:
def save_crit_eval(skill_name, criterion_name, verdict, reason, prompt_type, model_type, run_id):
    line = { "skill_name":skill_name, "criterion_name":criterion_name, "verdict":verdict, "reason":reason, "prompt_type":prompt_type, "model_type":model_type, "run_id":run_id, "timestamp": datetime.now().isoformat() }
    try:
        with open(f"prompt_{prompt_type}_model_{model_type}_autorubric_evaluation.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(line) + "\n")
    except IOError as e:
        print(f"Error saving criterion evaluation for {skill_name}: {e}")

def save_skill_total(skill_name, final_score, prompt_tokens, completion_tokens, total_tokens, completion_cost, prompt_type, model_type, run_id):
    line = {  "skill_name": skill_name, "final_score": final_score, "token_usage": {"prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens, "total_tokens": total_tokens}, "completion_cost":completion_cost, "prompt_type":prompt_type, "model_type":model_type, "run_id":run_id, "timestamp": datetime.now().isoformat() }
    try:
        with open(f"autorubric_stats_per_skill_prompt_{prompt_type}_model_{model_type}.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(line) + "\n")
    except IOError as e:
        print(f"Error saving skill total for {skill_name}: {e}")

def save_final_line(total_cost, total_score, total_prompt_tokens, total_completion_tokens, total_total_tokens, prompt_type, model_type, run_id):
    line = {"prompt_type":prompt_type, "model_type":model_type, "total_score":total_score, "token_usage": {"prompt_tokens": total_prompt_tokens, "completion_tokens": total_completion_tokens, "total_tokens": total_total_tokens}, "total_cost":total_cost, "run_id":run_id, "timestamp": datetime.now().isoformat()}
    try:
        with open(f"autorubric_evaluation_stats_per_prompt.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(line) + "\n")
    except IOError as e:
        print(f"Error saving final evaluation line: {e}")
    # total cost + total score + prompt_type + model_type

In [ ]:
# creation of the evaluation rubic
rubric = Rubric.from_dict(rubric_criteria)


async def evaluate_responses(rubric, prompt_type, model_type):

    run_id = str(uuid.uuid4())
    # Configure the grader
    responses = sorted(sample_responses(get_responses(f"/content/prompt_{prompt_type}_model_{model_type}.jsonl")),key=lambda x: x["output"]["json"]["skill_name"])

    grader = CriterionGrader(
        llm_config=LLMConfig(
        model="gpt-4.1-mini",
        temperature=0.0
        ),
        cannot_assess_config=CannotAssessConfig( strategy=CannotAssessStrategy.FAIL )
    )


    print("=" * 60)
    print("AGRI-SKILLS QUALITY ASSESSMENT")
    print("=" * 60)

    total_cost = 0.0
    total_score = 0.0
    total_prompt_tokens = 0
    total_completion_tokens = 0
    total_total_tokens = 0
    total_score = 0.0

    for i, response in enumerate(responses, 1):
        skill_name = response['output']['json']['skill_name']

        grade_result = await rubric.grade(
            to_grade=response['output']['text'],
            grader=grader,
            query=response['input']['prompt']
        )
        time.sleep(10)
        print(f"\n--- Prompt {i} evaluation concerning agricultural skill: {skill_name}" )

        print(f"Score: {grade_result.score:.2f}")
        total_score += grade_result.score

        if grade_result.completion_cost:
            total_cost += grade_result.completion_cost
        if grade_result.token_usage:
            total_prompt_tokens += grade_result.token_usage.prompt_tokens
            total_completion_tokens += grade_result.token_usage.completion_tokens
            total_total_tokens += grade_result.token_usage.total_tokens

        # Show per-criterion verdicts

        for criterion in grade_result.report:
            criterion_name = criterion.criterion.name
            if criterion.criterion.is_binary:
                verdict = criterion.final_verdict.name # Get 'MET' or 'UNMET'
                reason = criterion.final_reason
                print(f" {criterion_name}: {verdict} [{reason}] ")

            else:
                verdict = criterion.multi_choice_votes[0].value
                label = criterion.multi_choice_votes[0].selected_label
                reason = criterion.multi_choice_votes[0].reason
                print(f" {criterion_name}: {verdict} - {label} [{reason}] ")

            save_crit_eval(skill_name, criterion_name, verdict, reason, prompt_type, model_type, run_id)


        save_skill_total(skill_name, grade_result.score, grade_result.token_usage.prompt_tokens, grade_result.token_usage.completion_tokens, grade_result.token_usage.total_tokens, grade_result.completion_cost, prompt_type, model_type, run_id)

    print(f"\n{'=' * 60}")
    total_score = total_score / len(responses)
    save_final_line(total_cost, total_score, total_prompt_tokens, total_completion_tokens, total_total_tokens, prompt_type, model_type, run_id)
    print(f"Total evaluation cost: ${total_cost:.4f}")
    print("\n--- Stored Evaluation Results ---")


In [ ]:
await evaluate_responses(rubric, "3", "3")

AGRI-SKILLS QUALITY ASSESSMENT

--- Prompt 1 evaluation concerning agricultural skill: agricultural business management
Score: 1.00
 accuracy: 1.0 - Fully accurate and factually correct [The submission accurately defines agricultural business management as the application of business principles to agricultural production and marketing, aligning with the official ESCO description. It correctly identifies key processes such as financial analysis, strategic planning, logistics management, and pricing strategies, all relevant to the domain. The technical implications logically follow from these processes and reflect domain knowledge without introducing unsupported information.] 
 fluency: 1.0 - Fully fluent, grammatically correct, and natural-sounding text [The submission is fully fluent, grammatically correct, and natural-sounding, with no noticeable errors in syntax or grammar throughout the structured explanation.] 
 relevance: 1.0 - Fully relevant and focused on the query [The submissi

## Manual Evaluation

In order to conduct the manual evaluation, we need to creat readable PDFs with the responses and spreadsheets where the evaluator is going to store the scores.

## creation of easily readable pdfs for evaluators



In [ ]:
!pip install weasyprint

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 39.9 MB/s eta 0:00:00
  Attempting uninstall: tinycss2
    Found existing installation: tinycss2 1.4.0
    Uninstalling tinycss2-1.4.0:
      Successfully uninstalled tinycss2-1.4.0


In [ ]:
def prompt_1_template(output):
    concept = output.get("skill_concept", "N/A")

    template = "<span class='section-label'>Skill Concept:</span>"
    template += f"<div>{concept}</div>"

    template += "<span class='section-label'>Learning Path:</span>"
    template += "<ul>"
    for lrn in output.get("learning_path", []):
        title = lrn.get("step_title", "N/A")
        desc = lrn.get("step_description", "N/A")
        template += f"<li><span class='sub-title'>{title}:</span> {desc}</li>"
    template += "</ul>"

    return template


def prompt_2_template(output):
      template = "<span class='section-label'>Skill Applications:</span>"
      template += "<ul>"
      for app in output.get("skill_applications", []):
          title = app.get("application_title", "N/A")
          desc = app.get("application_description", "N/A")
          template += f"<li><span class='sub-title'>{title}:</span> {desc}</li>"
      template += "</ul>"

      sus = output.get("sustainability_connection", {})
      template += "<span class='section-label'>Sustainability Dimensions:</span>"
      template += "<ul>"
      template += f"<li><span class='sub-title'>Environmental Impact:</span> {sus.get('environmental_impact', 'N/A')}</li>"
      template += f"<li><span class='sub-title'>Resource Efficiency:</span> {sus.get('resource_efficiency', 'N/A')}</li>"
      template += f"<li><span class='sub-title'>Long-term Effects:</span> {sus.get('long_term_effects', 'N/A')}</li>"
      template += "</ul>"
      return template


def prompt_3_template(output):
      explanation = output.get("technical_explanation", {})
      template = "<span class='section-label'>Technical Explanation:</span>"
      template += f"<div><span class='sub-title'>Definition: </span>{explanation.get("definition","N/A")}</div>"
      template += f"<span class='sub-title'>Processes: </span>"
      template += "<ul>"
      for pr in explanation.get("processes", []):
          template += f"<li> {pr}</li>"
      template += "</ul>"
      template += f"<span class='sub-title'>Technical Implications: </span>"
      template += "<ul>"
      for tech in explanation.get("technical_implications", []):
          template += f"<li> {tech}</li>"
      template += "</ul>"

      return template

In [ ]:
import json
from weasyprint import HTML
import os

def generate_sampled_results_pdf(prompt_type, model_type):
    results_list = sample_responses(get_responses(f"/content/prompt_{prompt_type}_model_{model_type}.jsonl"))
    records = []

    # Process the list of JSON objects
    for data in results_list:
        # Extract the raw prompt from the input section
        raw_prompt = data.get("input", {}).get("prompt", "").strip()

        # The 'output' -> 'text' field in data is a stringified JSON
        output_text = data.get("output", {}).get("text", "{}")
        try:
            parsed_output = json.loads(output_text)
        except json.JSONDecodeError:
            parsed_output = {
                "skill_name": "Parsing Error",
                "skill_applications": [],
                "sustainability_connection": {}
            }

        # Extract skill name for sorting and display
        skill_name = parsed_output.get("skill_name", "Unknown Skill")

        records.append({
            "skill_name": skill_name,
            "prompt": raw_prompt,
            "output": parsed_output
        })

    # Sort alphabetically by skill name
    records.sort(key=lambda x: x["skill_name"].lower())
    # Build HTML Structure with CSS
    document_title = f"Prompt {prompt_type} - Model {model_type} - Sampled Results"

    html_template = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <style>
            @page {{ size: A4; margin: 2cm; }}
            body {{ font-family: 'Helvetica', 'Arial', sans-serif; line-height: 1.6; color: #333; }}
            h1 {{ text-align: center; color: #2e7d32; border-bottom: 2px solid #2e7d32; padding-bottom: 10px; }}
            .skill-section {{
                margin-bottom: 40px;
                page-break-inside: avoid;
                border: 1px solid #ddd;
                padding: 20px;
                border-radius: 8px;
                background-color: #fafafa;
            }}
            .skill-title {{ color: #1b5e20; font-size: 1.5em; margin-top: 0; text-transform: uppercase; }}
            .section-label {{ font-weight: bold; color: #388e3c; margin-top: 15px; display: block; border-bottom: 1px solid #ccc; }}
            .prompt-box {{
                font-style: italic;
                font-size: 0.85em;
                background: #f1f1f1;
                padding: 10px;
                border-left: 4px solid #888;
                margin: 10px 0;
                white-space: pre-wrap;
            }}
            ul {{ padding-left: 20px; }}
            li {{ margin-bottom: 8px; }}
            div {{ margin: 8px; }}
            .sub-title {{ font-weight: bold; color: #2c3e50; }}
        </style>
    </head>
    <body>
        <h1>{document_title}</h1>
    """

    for rec in records:
        output = rec["output"]
        html_template += f"<div class='skill-section'>"
        html_template += f"<h2 class='skill-title'>{rec['skill_name']}</h2>"

        # Prompt Box
        html_template += "<span class='section-label'>User Prompt Context:</span>"
        html_template += f"<div class='prompt-box'>{rec['prompt']}</div>"

        if prompt_type == "1": html_template += prompt_1_template(output)
        if prompt_type == "2": html_template += prompt_2_template(output)
        if prompt_type == "3": html_template += prompt_3_template(output)

        html_template += "</div>"

    html_template += "</body></html>"

    output_pdf = f"/content/results/prompt_{prompt_type}_model_{model_type}_sample_results.pdf"
    # Generate the PDF
    HTML(string=html_template).write_pdf(output_pdf)
    print(f"Report generated: {output_pdf}")



In [ ]:

for i in range(1,4):
  for j in range(1,4):
    generate_sampled_results_pdf(str(i), str(j))

NameError: name 'generate_sampled_results_pdf' is not defined

In [ ]:
generate_sampled_results_pdf("1", "2")

DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.002s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.005s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
INFO:fontTools.subset:kern dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:GPOS dropped
INFO:fontTools.subset:GSUB dropped
DEBUG:f

Report generated: /content/results/prompt_1_model_2_sample_results.pdf


## creation of the evaluation spreadshits for manual evaluation


In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation


def create_evaluation_spreadsheet(rubric_criteria, prompt_type, model_type):

    responses = sorted(sample_responses(get_responses(f"/content/prompt_{prompt_type}_model_{model_type}.jsonl")), key=lambda x: x["output"]["json"]["skill_name"])
    output_path = f"/content/spreadsheets/prompt_{prompt_type}_model_{model_type}_manual_evaluation.xlsx"

    final_rows = []
    for response in responses:
        for crit in rubric_criteria:
            final_rows.append({
                "Skill Name": response["output"]["json"]["skill_name"],
                "Criteria": crit['name'],
                "Requirement": crit["requirement"],
                "Type": crit["scale_type"] if "scale_type" in crit else "binary",
                "Score": "",        # Left blank for human evaluator
                "Justification": "" # Left blank for human evaluator
            })

    df = pd.DataFrame(final_rows)


    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            df.to_excel(writer, index=False, sheet_name='Grading')

            # Access the openpyxl objects
            worksheet = writer.sheets['Grading']

            # Define the two different validation rules with comma as decimal and semicolon as list separator
            dv_binary = DataValidation(type="list", formula1='"0;1"', allow_blank=True)
            dv_ordinal = DataValidation(type="list", formula1='"0;0,25;0,5;0,75;1"', allow_blank=True)

            # Add them to the worksheet
            worksheet.add_data_validation(dv_binary)
            worksheet.add_data_validation(dv_ordinal)

            for row_num in range(2, len(final_rows) + 2):
                cell_type = worksheet.cell(row=row_num, column=4).value
                score_cell = worksheet.cell(row=row_num, column=5)

                if cell_type == "binary":
                    dv_binary.add(score_cell)
                else:
                    dv_ordinal.add(score_cell)

In [ ]:
for i in range(1,4):
  for j in range(1,4):
    create_evaluation_spreadsheet(rubric_criteria,str(i), str(j))

In [ ]:
create_evaluation_spreadsheet(rubric_criteria,"1", "2")

### create unified spreadsheet for analysis
that contains the skills with theis categories and has columns for the llm and for the human as well


In [ ]:
import pandas as pd
from pprint import pprint

df = pd.read_csv("notion_evaluation_sample.csv")

skill_categories = dict(zip(df['Name'], df['Category']))

pprint(skill_categories)


{'agricultural business management': 'Agricultural Business Management',
 'agricultural equipment': 'Use and Maintainance of Equipment',
 'agroecology': 'Environmental Protection and Sustainability',
 'agronomical production principles': 'Crops and Land Cultivation',
 'animal production science': 'Livestock Management',
 'assign duties to agriculture workers': 'Agricultural Business Management',
 'crop production principles': 'Crops and Land Cultivation',
 'drive agricultural machines': 'Use and Maintainance of Equipment',
 'harvest crop': 'Crops and Land Cultivation',
 'manage crop production': 'Crops and Land Cultivation',
 'manage farm products': 'Agricultural Business Management',
 'manage livestock': 'Livestock Management',
 'operate agricultural machinery': 'Use and Maintainance of Equipment',
 'optimise production': 'Agricultural Business Management',
 'perform inspection analysis': 'Scientific Research and Knowledge',
 'prepare planting area': 'Crops and Land Cultivation',
 'pr

In [ ]:
import json
def get_llm_evaluation(prompt_type, model_type):
    llm_evaluation = {}
    with open(f"prompt_{prompt_type}_model_{model_type}_autorubric_evaluation.jsonl", 'r', encoding='utf-8') as f:
        for line in f:
          line = line.strip()
          if line:
              parsed_line = json.loads(line)
              llm_evaluation[f"{parsed_line['skill_name'].lower()}_{parsed_line['criterion_name']}"] = {"verdict":parsed_line['verdict'], "reason":parsed_line['reason']}
    return llm_evaluation

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation


def create_evaluation_spreadsheet(rubric_criteria, prompt_type, model_type):

    responses = sorted(sample_responses(get_responses(f"/content/prompt_{prompt_type}_model_{model_type}.jsonl")), key=lambda x: x["output"]["json"]["skill_name"].lower())
    output_path = f"/content/prompt_{prompt_type}_model_{model_type}_combined_evaluation.xlsx"

    llm_evaluation = get_llm_evaluation(prompt_type, model_type)

    final_rows = []
    for response in responses:
        for crit in rubric_criteria:
            final_rows.append({
                "Skill Name": response["output"]["json"]["skill_name"].lower(),
                "Category": skill_categories[response["output"]["json"]["skill_name"].lower()],
                "Criteria": crit['name'],
                "Requirement": crit["requirement"],
                "Type": crit["scale_type"] if "scale_type" in crit else "binary",
                "Manual Score": "",        # Left blank for human
                "LLM Score": llm_evaluation[f"{response["output"]["json"]["skill_name"].lower()}_{crit['name']}"]["verdict"],
                "Manual Justification": "", # Left blank for human
                "LLM Justification": llm_evaluation[f"{response["output"]["json"]["skill_name"].lower()}_{crit['name']}"]["reason"]
            })

    df = pd.DataFrame(final_rows)


    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            df.to_excel(writer, index=False, sheet_name='Grading')

            # Access the openpyxl objects
            worksheet = writer.sheets['Grading']

            # Define the two different validation rules with comma as decimal and semicolon as list separator
            dv_binary = DataValidation(type="list", formula1='"0,1"', allow_blank=True)
            dv_ordinal = DataValidation(type="list", formula1='"0,0.25,0.5,0.75,1"', allow_blank=True)

            # Add them to the worksheet
            worksheet.add_data_validation(dv_binary)
            worksheet.add_data_validation(dv_ordinal)

            for row_num in range(2, len(final_rows) + 2):
                cell_type = worksheet.cell(row=row_num, column=5).value
                score_cell = worksheet.cell(row=row_num, column=6)

                if cell_type == "binary":
                    dv_binary.add(score_cell)
                else:
                    dv_ordinal.add(score_cell)

In [ ]:

create_evaluation_spreadsheet(rubric_criteria,"1","3")






